# TFM — Parque de Vehículos (DGT) vs. Renta de los Hogares (INE)
## Fase 1 — Limpieza profunda y Feature Engineering (ETL / Data Wrangling)

Partiendo de `dataset_fusionado.parquet` (salida de la Fase 0), este notebook realiza la
limpieza profunda, la **decodificación de variables con los diccionarios oficiales de la
DGT** y la ingeniería de características, dejando el dataset listo para el EDA.

**Orden del pipeline:** perfilado → duplicados → nombres de columna → *type casting* →
selección → gestión de nulos → estandarización de texto → decodificación de códigos →
*feature engineering* → guardado.

> 🟢 **Buenas prácticas aplicadas en todo el notebook:**
> - **Vectorización**: sin bucles fila a fila ni `apply`/`lambda` sobre los 6,6 M de
>   registros; se usan `.str`, `.map(dict)`, `np.where`, `pd.qcut`, aritmética nativa.
> - **Carga defensiva**: lectura envuelta en `try/except` con verificación de ruta.
> - **Optimización de memoria**: tipos adecuados (`category`, `Int8/16`, `float32`).

---
## 0. Configuración e importaciones

In [1]:
%reload_ext autoreload
%autoreload 2

# ==============================================================================
# 1. LIBRERÍAS DE LA BILIOTECA ESTÁNDAR (System & File System)
# ==============================================================================
import sys
from pathlib import Path

# ==============================================================================
# 2. LIBRERÍAS DE TERCEROS (Data Science & Analytics)
# ==============================================================================
import numpy as np
import pandas as pd

# ==============================================================================
# 3. CONFIGURACIÓN DE RUTAS Y ENTORNO
# ==============================================================================
# Ruta base donde se guardarán los datasets ya limpios y transformados
DIR_PROCESSED = Path("../data/processed")

# Añadir el directorio de código fuente ('src') al PATH para poder importar módulos locales
sys.path.append("../src")

# ==============================================================================
# 4. MÓDULOS PROPIOS (Trabajo de Fin de Máster - TFM)
# ==============================================================================
import tfm_limpieza  as tl      # Funciones para el preprocesamiento y limpieza de datos
import tfm_io        as ti      # Funciones de lectura, escritura y gestión de archivos (I/O)
import tfm_eda       as te      # Funciones para el Análisis Exploratorio de Datos



---
## 1. Carga del dataset fusionado (con control de errores)

Se parte del fichero generado en la Fase 0. La lectura se envuelve en `try/except` con
verificación de ruta; se usa `raise SystemExit` (no `exit()`) para detener de forma
limpia sin matar el kernel de Jupyter.

In [2]:
# Cargar el dataset fusionado
df = ti.cargar_parquet("dataset_fusionado.parquet", DIR_PROCESSED)


Cargado exitosamente: 6,613,034 filas x 55 columnas


---
## 2. Perfilado inicial

Foto de partida: dimensiones, tipos, nulos, duplicados y estadísticos.

In [3]:
te.eda_preliminar(df, cols_notin_outlier=["anio_matriculacion"])


>>> INFO ESTRUCTURAL
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6613034 entries, 0 to 6613033
Data columns (total 55 columns):
 #   Column                                     Dtype         
---  ------                                     -----         
 0   PROVINCIA                                  object        
 1   MUNICIPIO                                  object        
 2   FABRICANTE                                 object        
 3   MARCA                                      object        
 4   MODELO                                     object        
 5   TIPO                                       object        
 6   VARIANTE                                   object        
 7   VERSION                                    object        
 8   PROVINCIA_MATR                             object        
 9   FECHA_MATR                                 datetime64[ns]
 10  FEC_PRIM_MATR                              datetime64[ns]
 11  CLASE_MATR                               

,PROVINCIA,MUNICIPIO,FABRICANTE,MARCA,MODELO,TIPO,VARIANTE,VERSION,PROVINCIA_MATR,FECHA_MATR,...,nombre_provincia,cod_municipio,nombre_municipio,Media de la renta por unidad de consumo,Mediana de la renta por unidad de consumo,Renta bruta media por hogar,Renta bruta media por persona,Renta neta media por hogar,Renta neta media por persona,tiene_municipio
5102955,28,28045,None,HYUNDAI,None,None,None,None,28,2012-02-22,...,Madrid,28045,Colmenar Viejo,27567.0,24150.0,65814.0,22447.0,52003.0,17736.0,True
4403345,28,28013,HYUNDAI MOTOR MANUFACTURING CZECH S.R.O,HYUNDAI,I30,GDH,B5D51,None,47,2015-03-25,...,Madrid,28013,Aranjuez,21662.0,19250.0,49015.0,17173.0,40453.0,14173.0,True
4760958,28,28148,AUTOMOBILE DACIA SA,DACIA,None,SD,None,None,28,2012-06-27,...,Madrid,28148,Torrejón de Ardoz,22129.0,19950.0,50089.0,17473.0,41459.0,14463.0,True
4604607,28,28068,PSA PEUGEOT-CITROEN S.A,OPEL,None,None,None,None,28,2025-01-30,...,Madrid,28068,Guadarrama,24913.0,22050.0,55113.0,20395.0,44863.0,16602.0,True
1173645,08,None,AUTOMOBILES PEUGEOT-TALBOT SA,PEUGEOT,None,L,None,None,05,2021-04-30,...,Barcelona,None,None,NaN,NaN,NaN,NaN,NaN,NaN,False



>>> DIMENSIONES


'El conjunto de datos tiene 6,613,034 filas y 55 columnas.'


>>> VALORES NULOS (%)


...

nombre_municipio
Madrid                16.66
Alcobendas             6.07
Barcelona              5.52
València               3.83
Sevilla                3.35
Majadahonda            3.29
Alacant/Alicante       1.78
Boadilla del Monte     1.41
Moralzarzal            1.38
Elx/Elche              1.25
Name: proportion, dtype: float64


>>> DETECCIÓN DE POSIBLES OUTLIERS (MÉTODO IQR)


,Cantidad de Outliers,% del Total
Mediana de la renta por unidad de consumo,401636,6.07
Renta neta media por hogar,394995,5.97
Renta bruta media por hogar,389761,5.89
Media de la renta por unidad de consumo,113102,1.71
Renta bruta media por persona,35246,0.53
Renta neta media por persona,35246,0.53



>>> TENDENCIA CENTRAL Y DISPERSIÓN (NUMÉRICAS)


,count,mean,min,25%,50%,75%,max,std
FECHA_MATR,6613034,2019-06-29 05:24:49.469071360,2010-01-04 00:00:00,2016-01-28 00:00:00,2019-08-09 00:00:00,2023-09-13 00:00:00,2026-03-31 00:00:00,NaN
FEC_PRIM_MATR,6613034,2019-04-03 02:42:57.077210880,2010-01-01 00:00:00,2015-10-02 00:00:00,2019-04-30 00:00:00,2023-06-20 00:00:00,2026-03-31 00:00:00,NaN
Media de la renta por unidad de consumo,5502832.0,25740.016112,14207.0,21168.0,23739.0,29129.0,49811.0,6365.124622
Mediana de la renta por unidad de consumo,5502832.0,21656.36614,12250.0,18550.0,21350.0,23450.0,37450.0,4413.372369
Renta bruta media por hogar,5502832.0,58795.218769,28157.0,45949.0,52807.0,64574.0,138851.0,19810.773353
Renta bruta media por persona,5502832.0,21596.846109,10972.0,16862.0,20310.0,25397.0,44016.0,5931.412175
Renta neta media por hogar,5502832.0,46114.646451,24186.0,37940.0,42616.0,49916.0,96290.0,13069.461216
Renta neta media por persona,5502832.0,16973.763584,9825.0,14045.0,16121.0,19632.0,30524.0,3810.019712


---
## 3. Duplicados — decisión: NO eliminar

El microdato de la DGT **no contiene matrícula ni identificador único** de vehículo. Dos
registros con ficha técnica idéntica son **dos vehículos reales distintos**, no una
duplicación de datos. Eliminarlos subestimaría el tamaño real del parque. La unidad de
análisis es *vehículo-registro*, por lo que se **conservan**.

In [4]:
n_dup = df.duplicated().sum()
print(f"Filas con ficha tecnica identica: {n_dup:,} ({n_dup/len(df)*100:.2f}%)")
print("Decision: se CONSERVAN (vehiculos reales distintos; la DGT no aporta ID unico).")

Filas con ficha tecnica identica: 928,580 (14.04%)
Decision: se CONSERVAN (vehiculos reales distintos; la DGT no aporta ID unico).


---
## 4. Estandarización de nombres de columna

Primero se renombran a mano las 6 columnas de renta del INE (nombres largos) y
**después** se aplica la función genérica (minúsculas, sin espacios). El orden importa:
si se invierte, los nombres largos del INE no coincidirían con el diccionario de
renombrado.

In [5]:
# 1) Renombrado especifico de las columnas de renta del INE
df = df.rename(columns={
    "Media de la renta por unidad de consumo":   "renta_media_uc",
    "Mediana de la renta por unidad de consumo": "renta_mediana_uc",
    "Renta bruta media por hogar":               "renta_bruta_hogar",
    "Renta bruta media por persona":             "renta_bruta_persona",
    "Renta neta media por hogar":                "renta_neta_hogar",
    "Renta neta media por persona":              "renta_neta_persona",
})

# 2) Estandarizacion generica del resto (minusculas, guiones bajos)
df = tl.estandarizar_nombres_columnas(df)

print(df.columns.tolist())

['provincia', 'municipio', 'fabricante', 'marca', 'modelo', 'tipo', 'variante', 'version', 'provincia_matr', 'fecha_matr', 'fec_prim_matr', 'clase_matr', 'procedencia', 'nuevo_usado', 'tipo_titular', 'num_titulares', 'subtipo_dgt', 'tipo_dgt', 'cat_euro', 'clas_construccion', 'clas_utilizacion', 'servicio', 'renting', 'tara', 'peso_max', 'mom', 'mmta', 'cilindrada', 'potencia', 'kw', 'propulsion', 'catelect', 'consumo', 'autonomia', 'alimentacion', 'tipo_distintivo', 'emisiones_euro', 'emisiones_co2', 'carroceria', 'distancia_ejes', 'eje_anterior', 'eje_posterior', 'plazas', 'plazas_max', 'plazas_pie', 'nombre_provincia', 'cod_municipio', 'nombre_municipio', 'renta_media_uc', 'renta_mediana_uc', 'renta_bruta_hogar', 'renta_bruta_persona', 'renta_neta_hogar', 'renta_neta_persona', 'tiene_municipio']


---
## 5. Type casting (optimización de memoria)

Conversión explícita de tipos: decimales con coma española a `float32`, enteros a tipos
*nullable* (`Int8`/`Int16`) y cualitativas a `category`.

> 🟢 **`potencia` vs `kw`.** La columna `potencia` de la DGT es la **potencia fiscal**
> (CVF), no la del motor. La potencia real está en `kw` (de ahí se derivará
> `potencia_cv`). `potencia` se descarta en la selección.

In [6]:
# Decimales (coma española -> punto) a float32
cols_decimales = ["kw", "consumo", "emisiones_co2", "autonomia"]
df = tl.corregir_decimales(df, cols_decimales)

# Enteros nullable
config_enteros = {
    "cilindrada":    "Int16",
    "num_titulares": "Int8",
    "puertas":       "Int8"  
}
df = tl.corregir_enteros_nullable(df, config_enteros)


# Cualitativas -> category
cols_categoria = [
    "provincia", "nombre_provincia", "municipio", "cod_municipio", "nombre_municipio",
    "marca", "modelo", "propulsion", "catelect", "alimentacion", "tipo_distintivo",
    "carroceria", "cat_euro", "emisiones_euro", "tipo_titular", "renting",
    "nuevo_usado", "procedencia",
]

# Funcion generica que ademas muestra resumen de categorias
df = tl.a_categoria(df,cols_categoria)
    
df.dtypes

provincia                    category
municipio                    category
fabricante                     object
marca                        category
modelo                       category
tipo                           object
variante                       object
version                        object
provincia_matr                 object
fecha_matr             datetime64[ns]
fec_prim_matr          datetime64[ns]
clase_matr                     object
procedencia                  category
nuevo_usado                  category
tipo_titular                 category
num_titulares                    Int8
subtipo_dgt                    object
tipo_dgt                       object
cat_euro                     category
clas_construccion              object
clas_utilizacion               object
servicio                       object
renting                      category
tara                           object
peso_max                       object
mom                            object
mmta        

---
## 6. Selección de columnas

Se descartan las variables **sin valor analítico** para las hipótesis (masas, ejes,
plazas, potencia fiscal, identificadores técnicos redundantes y campos administrativos
de matriculación). El criterio es la **irrelevancia analítica**, no el número de nulos.

> 🟢 **`catelect` se CONSERVA.** El documento oficial de interfaz revela que `catelect`
> (categoría eléctrica: BEV/PHEV/HEV/REEV/FCEV) es la variable que codifica la
> electrificación, clave para H3. Su ~75 % de nulos es **estructural** (un vehículo de
> combustión no tiene categoría eléctrica), no un defecto de calidad.

In [7]:
cols_descartar = [
    "tara", "peso_max", "mom", "mmta", "distancia_ejes", "eje_anterior",
    "eje_posterior", "plazas", "plazas_max", "plazas_pie", "potencia",
    "version", "variante", "tipo", "subtipo_dgt", "clas_construccion",
    "clas_utilizacion", "fecha_matr", "provincia_matr", "clase_matr",
    "servicio", "tipo_dgt", "fabricante",
]
df = df.drop(columns=[c for c in cols_descartar if c in df.columns])
print(f"Columnas tras seleccion: {df.shape[1]}")
print(df.columns.tolist())

Columnas tras seleccion: 32
['provincia', 'municipio', 'marca', 'modelo', 'fec_prim_matr', 'procedencia', 'nuevo_usado', 'tipo_titular', 'num_titulares', 'cat_euro', 'renting', 'cilindrada', 'kw', 'propulsion', 'catelect', 'consumo', 'autonomia', 'alimentacion', 'tipo_distintivo', 'emisiones_euro', 'emisiones_co2', 'carroceria', 'nombre_provincia', 'cod_municipio', 'nombre_municipio', 'renta_media_uc', 'renta_mediana_uc', 'renta_bruta_hogar', 'renta_bruta_persona', 'renta_neta_hogar', 'renta_neta_persona', 'tiene_municipio']


---
## 7. Gestión de nulos

No se imputa de forma genérica: cada decisión responde al **significado** del nulo y a su
impacto en las hipótesis.

| Bloque | Columnas | Estrategia | Justificación |
|---|---|---|---|
| 1. Territorio/renta | `municipio`, `cod_municipio`, `nombre_municipio`, 6× `renta_*` | **Conservar NaN** + bandera `tiene_municipio` | El hueco son municipios <10.000 hab. (la DGT suprime el municipio por anonimización). Imputar renta sería inventar el dato. La provincia sí está al 100 %. |
| 2. Categóricas descriptivas | `modelo`, `marca`, `carroceria`, `cat_euro`, `emisiones_euro` | **Categoría `"DESCONOCIDO"`** | Hacer el nulo visible como nivel propio en vez de imputar la moda. Sus nulos son valores enmascarados por anonimización (ocurrencia ≤5), no datos perdidos. |
| 3. Numéricas ambientales | `emisiones_co2`, `autonomia`, `consumo` | **Conservar NaN** | No imputar media: sesgaría H3. El NaN es "no aplica / no informado". |
| 4. Ceros en `cilindrada`/`kw` | — | **Diferir a outliers (EDA)** | Un 0 puede ser eléctrico legítimo (0 cc) o nulo encubierto; se distingue cruzando con la propulsión en la fase de anomalías. |

> 🟢 Origen de los huecos (según el documento oficial de interfaz de la DGT): los nulos de
> `marca`/`modelo` son **valores enmascarados** (marcas/modelos con ≤5 ocurrencias,
> sustituidos por el carácter `¡`); los de `municipio`/renta provienen de la **supresión
> del municipio en poblaciones <10.000 habitantes**. En ambos casos el nulo es
> estructural.

In [8]:
tl.resumen_nulos(df).query("pct_nulos > 0")

,columna,n_nulos,pct_nulos
0,catelect,4984344,75.37
1,modelo,1158168,17.51
2,municipio,1110202,16.79
3,renta_bruta_hogar,1110202,16.79
4,renta_bruta_persona,1110202,16.79
5,renta_neta_hogar,1110202,16.79
6,renta_neta_persona,1110202,16.79
7,cod_municipio,1110202,16.79
8,nombre_municipio,1110202,16.79
9,renta_media_uc,1110202,16.79


> 🟢 **Bloque 1 — Territorio/renta: conservar.** Se verifica que el NaN de renta coincide
> 1:1 con la ausencia de municipio (la bandera basta para filtrar de forma limpia).

In [9]:
cols_renta = [c for c in df.columns if c.startswith("renta")]
coincide = (df[cols_renta[0]].isna() == ~df["tiene_municipio"]).all()
print(f"NaN de renta <=> sin municipio: {coincide}")
print(f"Vehiculos sin municipio (<10k hab.): {(~df['tiene_municipio']).sum():,} "
      f"({(~df['tiene_municipio']).mean()*100:.1f}%)")

NaN de renta <=> sin municipio: True
Vehiculos sin municipio (<10k hab.): 1,110,202 (16.8%)


> 🟢 **Bloque 2 — Categóricas descriptivas: `"DESCONOCIDO"`.** En dtype `category` hay que
> registrar el nuevo nivel antes de rellenar. Vectorizado (bucle sobre columnas, no sobre
> filas).

In [10]:
cols_desconocido = ["modelo", "marca", "carroceria", "cat_euro", "emisiones_euro"]
for col in cols_desconocido:
    if "DESCONOCIDO" not in df[col].cat.categories:
        df[col] = df[col].cat.add_categories(["DESCONOCIDO"])
    df[col] = df[col].fillna("DESCONOCIDO")
df[cols_desconocido].isna().sum()

modelo            0
marca             0
carroceria        0
cat_euro          0
emisiones_euro    0
dtype: int64

> 🟢 **Bloque 3 — Numéricas ambientales: conservar NaN.** No se imputa la media (sesgaría
> H3). Los ceros (p. ej. `autonomia == 0` en combustión) se tratan como outliers en el EDA.
>
> **Nota de interpretación (doc. oficial).** `consumo` y `autonomia` son métricas
> **eléctricas** (consumo Wh/km y autonomía eléctrica km, WLTP), no consumo de
> combustible. Solo tienen valor real en electrificados. La métrica ambiental transversal
> a todos los vehículos es `emisiones_co2` (g/km, WLTP).

In [11]:
tl.resumen_nulos(df).query("n_nulos > 0")

,columna,n_nulos,pct_nulos
0,catelect,4984344,75.37
1,municipio,1110202,16.79
2,renta_mediana_uc,1110202,16.79
3,renta_bruta_hogar,1110202,16.79
4,renta_bruta_persona,1110202,16.79
5,renta_neta_hogar,1110202,16.79
6,renta_neta_persona,1110202,16.79
7,cod_municipio,1110202,16.79
8,nombre_municipio,1110202,16.79
9,renta_media_uc,1110202,16.79


---
## 8. Estandarización de texto

`marca` venía limpia de formato. `modelo` tenía variantes de acentuación: se normaliza
operando sobre las **categorías** (no sobre los 6,6 M de filas) y se propaga con `.map`.
Además se construye un **diccionario de alias** para unificar sinónimos de marca (mismo
fabricante escrito de varias formas) — la capa de gobernanza que pide el enunciado.

In [12]:
# modelo: normalizacion de formato sobre las categorias (no sobre los 6,6M de filas)
map_modelo = dict(zip(
    df["modelo"].cat.categories,
    df["modelo"].cat.categories.str.strip().str.upper()
))
df["modelo"] = df["modelo"].map(map_modelo).astype("category")
print(f"modelo: {df['modelo'].cat.categories.size:,} categorias")

modelo: 5,133 categorias


In [13]:
# Diccionario de alias de marca (sinonimos del mismo fabricante).
# Decisiones de criterio comentadas. Las marcas ambiguas (DAIMLER, DAIMLER CHRYSLER y
# JAGUAR LAND ROVER LIMITED) se dejan SIN fundir a proposito.
alias_marca = {
    # Fusiones claras (variantes de escritura del mismo fabricante)
    "MERCEDES BENZ":                 "MERCEDES-BENZ",
    "VOLKSWAGEN AG":                 "VOLKSWAGEN",
    "VOLKSWAGEN V W":                "VOLKSWAGEN",
    "VOLKSWAGEN VW":                 "VOLKSWAGEN",
    "VOLKSWAGEN, VW":                "VOLKSWAGEN",
    "TESLA MOTORS":                  "TESLA",
    "DS AUTOMOBILES":                "DS",
    "MG ROEWE":                      "MG",
    "MG,ROEWE":                      "MG",
    "AUTOMOBILI LAMBORGHINI S.P.A.": "LAMBORGHINI",
    "FORD-CNG-TECHNIK":              "FORD",
    # Decisiones de criterio
    "MERCEDES-AMG":                  "MERCEDES-BENZ",  # submarca deportiva de Mercedes
    "BMW I":                         "BMW",            # submarca electrica (se mide con catelect)
    "VAUXHALL":                      "OPEL",           # marca britanica de Opel, mismos coches
    "SIN MARCA":                     "DESCONOCIDO",    # unificar los dos cubos de "sin marca"
}
s = df["marca"].astype(object)
df["marca"] = s.map(alias_marca).fillna(s).astype("category")
print(f"marca: {df['marca'].cat.categories.size} categorias (antes 151)")

marca: 136 categorias (antes 151)


---
## 9. Decodificación de variables codificadas (diccionarios oficiales DGT)

Varias columnas son **códigos**. Se traducen a etiquetas legibles con el *Documento de
interfaz de salida — Fichero de Parque Anual* de la DGT. Los códigos de "sin informar" se
unifican con `DESCONOCIDO`.

> 🟢 Detalle clave del diccionario de **propulsión**: `6 = GLP` y `7 = GNC` son **gas, no
> híbridos**; `D` y `G` son variantes de Diésel y Gasolina. La electrificación NO está en
> `propulsion` (que codifica el combustible) sino en `catelect`.

In [14]:
# ALIMENTACION: 0=Sin informar (-> DESCONOCIDO), B=Bifuel, F=Flexifuel, M=Monofuel
COD_ALIMENTACION = {"M": "MONOFUEL", "B": "BIFUEL", "F": "FLEXIFUEL", "0": "DESCONOCIDO"}
df["alimentacion"] = df["alimentacion"].astype(object).replace(COD_ALIMENTACION)
df["alimentacion"] = df["alimentacion"].fillna("DESCONOCIDO").astype("category")

# PROCEDENCIA: 0=Nacional, 1=Import no UE, 2=Subasta, 3=Import UE
COD_PROCEDENCIA = {"0": "NACIONAL", "1": "IMPORT NO UE", "2": "SUBASTA", "3": "IMPORT UE"}
df["procedencia"] = df["procedencia"].astype(object).replace(COD_PROCEDENCIA)
df["procedencia"] = df["procedencia"].fillna("DESCONOCIDO").astype("category")

# RENTING: N=No, S=Si
df["renting"] = df["renting"].astype(object).replace({"N": "NO", "S": "SI"})
df["renting"] = df["renting"].fillna("DESCONOCIDO").astype("category")

# NUEVO_USADO: N=Nuevo, U=Usado
df["nuevo_usado"] = df["nuevo_usado"].astype(object).replace({"N": "NUEVO", "U": "USADO"})
df["nuevo_usado"] = df["nuevo_usado"].fillna("DESCONOCIDO").astype("category")

for c in ["alimentacion", "procedencia", "renting", "nuevo_usado"]:
    print(f"{c:13} -> {df[c].cat.categories.tolist()}")

alimentacion  -> ['BIFUEL', 'DESCONOCIDO', 'FLEXIFUEL', 'MONOFUEL']
procedencia   -> ['DESCONOCIDO', 'IMPORT NO UE', 'IMPORT UE', 'NACIONAL', 'SUBASTA']
renting       -> ['DESCONOCIDO', 'NO', 'SI']
nuevo_usado   -> ['NUEVO', 'USADO']


---
## 10. Feature engineering

Variables derivadas para responder a las hipótesis. Todo vectorizado.

In [15]:
# Fecha de referencia = foto de los microdatos DGT (marzo 2026).
# NO se usa datetime.now(): la antiguedad debe medirse respecto a la foto de los datos.
FECHA_REF = pd.Timestamp("2026-03-31")

df["anio_matriculacion"] = df["fec_prim_matr"].dt.year.astype("Int16")
df["antiguedad"]    = ((FECHA_REF - df["fec_prim_matr"]).dt.days / 365.25).astype("float32")
df["potencia_cv"]   = (df["kw"] * 1.35962).astype("float32")   # 1 kW = 1.35962 CV

df[["anio_matriculacion", "antiguedad", "potencia_cv"]].describe().round(2)

,anio_matriculacion,antiguedad,potencia_cv
count,6613034.0,6613034.00,6613032.00
mean,2018.76,6.99,126.18
std,4.64,4.64,51.49
min,2010.0,0.00,0.00
25%,2015.0,2.78,97.89
50%,2019.0,6.92,115.57
75%,2023.0,10.49,149.56
max,2026.0,16.24,1359.61


In [16]:
# tipo_combustible: decodificacion de PROPULSION (diccionario oficial DGT)
COD_PROPULSION = {
    "0": "GASOLINA", "1": "DIESEL", "2": "ELECTRICO", "3": "OTROS", "4": "BUTANO",
    "5": "SOLAR", "6": "GLP", "7": "GNC", "8": "GNL", "9": "HIDROGENO",
    "A": "BIOMETANO", "B": "ETANOL", "C": "BIODIESEL", "D": "DIESEL", "G": "GASOLINA",
}
df["tipo_combustible"] = (df["propulsion"].astype(object)
                          .map(COD_PROPULSION).fillna("DESCONOCIDO").astype("category"))
df["tipo_combustible"].value_counts(dropna=False)

tipo_combustible
GASOLINA       3819526
DIESEL         2450248
ELECTRICO       207596
GLP             122094
GNC              13155
DESCONOCIDO        178
HIDROGENO           71
GNL                 56
ETANOL              35
BIODIESEL           33
BUTANO              33
BIOMETANO            9
Name: count, dtype: int64

In [17]:
# Electrificacion (campo oficial CATELECT: BEV/PHEV/HEV/REEV/FCEV).
df["es_electrificado"]     = df["catelect"].notna()
df["tipo_electrificacion"] = (df["catelect"].astype(object)
                              .fillna("COMBUSTION").astype("category"))
print(df["es_electrificado"].value_counts())
print(df["tipo_electrificacion"].value_counts())

es_electrificado
False    4984344
True     1628690
Name: count, dtype: int64
tipo_electrificacion
COMBUSTION    4984344
HEV           1198745
PHEV           232344
BEV            195972
REEV             1596
FCEV               33
Name: count, dtype: int64


> 🟢 **Por qué NO se crea una variable `premium`.** Se valoró una variable booleana de
> marca *premium* para contrastar H4 y se descarta de forma deliberada:
>
> 1. **No es objetiva.** "Premium" es una segmentación comercial, no una categoría del
>    dato; etiquetarla a criterio propio introduciría sesgo del autor.
> 2. **Generaría circularidad.** Definirla por atributos técnicos (potencia, cilindrada) y
>    "demostrar" luego que las premium tienen más cilindrada sería una tautología, dado que
>    esas variables son objeto de análisis en H1.
>
> En su lugar, la gama de marca se trata como **resultado emergente**: en el EDA se analiza
> la renta del territorio por marca y se deja que el patrón de concentración surja de los
> datos, sin imponer una clasificación previa.

In [ ]:
# Definimos los 5 tramos
etiquetas = ["Muy baja", "Baja", "Media", "Alta", "Muy alta"]

# 1. Aislar los municipios únicos con su renta correspondiente
renta_muni = (df.loc[df["tiene_municipio"], ["cod_municipio", "renta_neta_persona"]]
                .astype({"cod_municipio": "object"})
                .drop_duplicates("cod_municipio"))

# 2. Calcular los quintiles (5 tramos) basados únicamente en los municipios
renta_muni["segmento_renta"] = pd.qcut(
    renta_muni["renta_neta_persona"], q=5, labels=etiquetas, duplicates="drop"
)

# 3. Crear una serie-diccionario para trasladar el dato de vuelta
mapa_seg = renta_muni.set_index("cod_municipio")["segmento_renta"]

# 4. Propagar el segmento a todos los coches y clasificar los nulos
df["segmento_renta"] = (df["cod_municipio"].astype(object).map(mapa_seg)
                        .astype(object).fillna("SIN DATO").astype("category"))

# 5. Convertir a variable categórica ordinal para mantener el orden
df["segmento_renta"] = df["segmento_renta"].cat.set_categories(
    etiquetas + ["SIN DATO"], ordered=True
)

print("Municipios por quintil:", renta_muni["segmento_renta"].value_counts().to_dict())
print(df["segmento_renta"].value_counts(dropna=False))

Municipios por quintil: {'Muy baja': 55, 'Media': 55, 'Muy alta': 55, 'Baja': 54, 'Alta': 54}
segmento_renta
Muy alta    2418559
SIN DATO    1110202
Media       1009945
Alta         972820
Baja         576500
Muy baja     525008
Name: count, dtype: int64


> 🟢 **Por qué los quintiles se calculan por municipio y no por vehículo.** La renta es un
> dato del municipio, no del coche. Si formara los cinco grupos contando vehículos, Madrid
> capital —con millones de coches que comparten un único valor de renta— arrastraría los
> cortes hacia su nivel y aplastaría al resto de territorios. Calculando los quintiles sobre
> los municipios (cada uno cuenta una vez) y propagándolos luego a sus vehículos, cada tramo
> agrupa una quinta parte de los municipios, que es lo coherente para comparar territorios
> por renta.

---
## 11. Limpieza final y guardado

Se descartan los **códigos crudos** `propulsion` y `catelect`, ya redundantes con sus
versiones legibles (`tipo_combustible`, `tipo_electrificacion` / `es_electrificado`). Se
verifica el resultado y se guarda el **dataset final procesado**, entregable del
proyecto.

In [19]:
df = df.drop(columns=["propulsion", "catelect"])

print(f"Dataset final: {df.shape[0]:,} filas x {df.shape[1]} columnas\n")
print("Tipos:")
print(df.dtypes.to_string())
print("\nNulos restantes (por diseño):")
print(tl.resumen_nulos(df).query("n_nulos > 0").to_string(index=False))

Dataset final: 6,613,034 filas x 37 columnas

Tipos:
provincia                     category
municipio                     category
marca                         category
modelo                        category
fec_prim_matr           datetime64[ns]
procedencia                   category
nuevo_usado                   category
tipo_titular                  category
num_titulares                     Int8
cat_euro                      category
renting                       category
cilindrada                       Int16
kw                             float32
consumo                        float32
autonomia                      float32
alimentacion                  category
tipo_distintivo               category
emisiones_euro                category
emisiones_co2                  float32
carroceria                    category
nombre_provincia              category
cod_municipio                 category
nombre_municipio              category
renta_media_uc                 float64
renta_media

In [ ]:
ruta_out = DIR_PROCESSED / "dataset_procesado.parquet"
ti.guardar_parquet(df, ruta_out)

Guardado: ..\data\processed\dataset_procesado.parquet  (6,613,034 filas, 131.6 MB)


---
## Resumen de la Fase 1

| Requisito mínimo del TFM | Exigido | Conseguido |
|---|---|---|
| Filas | 50.000 | **6.613.034** |
| Columnas con valor analítico | 20 | **37** |
| Tipos de dato mixtos | num / cat / fecha | Presentes |
| Variables derivadas (feature engineering) | — | antigüedad, potencia_cv, tipo_combustible, electrificación, segmento_renta, año |

**Variables nuevas creadas:** `anio_matriculacion`, `antiguedad`, `potencia_cv`,
`tipo_combustible`, `es_electrificado`, `tipo_electrificacion`, `segmento_renta`.

**Próxima fase — EDA (`03_eda.ipynb`).** Sobre `dataset_procesado.parquet`: perfilado
estadístico, distribuciones, detección de outliers (incl. los ceros de `kw`/`cilindrada`
y el `kw=999` placeholder) y matriz de correlaciones, argumentando el impacto de la
**colinealidad** (p. ej. `kw`–`cilindrada`, y las 6 variables de renta entre sí) de cara
a un eventual modelo predictivo.